# Corrected Hybrid GCN Ablation Study

This notebook corrects the original ablation implementation so that:

- **CNN only** and **CNN + PCA** are genuinely different configurations.
- **GCN raw** uses the original CNN feature dimension.
- **GCN + PCA** uses PCA fitted **only on the training features**.
- Validation and test metrics are calculated **only on the corresponding held-out nodes**.
- Graph construction uses **features only** and does **not use ground-truth class labels**, avoiding label leakage.
- Class weighting is applied **only to training nodes**.
- The same train/validation/test split is used for all ablations.
- Results include validation accuracy, test accuracy, Macro-F1, and per-class test performance.

The graph used for the ablation is fixed to a feature-based kNN graph so that the ablation isolates the contribution of PCA, GCN learning, and class weighting rather than changing graph topology at the same time.


In [ ]:
import os, glob, gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

PCA_DIM = 256
K_GRAPH = 12

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.30

LR_HEAD = 1e-3
EPOCHS_HEAD = 100

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:
# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.asarray(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", class_names)
print("Class counts:", dict(zip(class_names, np.bincount(labels, minlength=num_classes))))


In [ ]:
# ======================
# FIXED STRATIFIED SPLIT
# 70% train / 20% validation / 10% test
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=np.float32)
mask_va = np.zeros(N, dtype=np.float32)
mask_te = np.zeros(N, dtype=np.float32)

mask_tr[idx_tr] = 1.0
mask_va[idx_va] = 1.0
mask_te[idx_te] = 1.0

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


In [ ]:
# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label

def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label

def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))
    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False)
all_ds = make_ds(np.arange(N), training=False)


In [ ]:
# ======================
# CNN FEATURE EXTRACTOR
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn_model = Model(inp, out)
    backbone = Model(inp, feat)
    return cnn_model, backbone

cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

class_weights_dict = dict(enumerate(
    compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=labels[idx_tr]
    )
))

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    class_weight=class_weights_dict,
    callbacks=[EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    )],
    verbose=1
)


In [ ]:
# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []
    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())
    return np.vstack(X).astype(np.float32), np.concatenate(Y)

X_all, y_all = extract_features(all_ds)

print("CNN feature matrix:", X_all.shape)


In [ ]:
# ======================
# FEATURE TRANSFORMATION
# IMPORTANT:
# PCA is fitted ONLY on training data.
# ======================
def make_features(use_pca):
    X_tr0 = X_all[idx_tr]
    X_va0 = X_all[idx_va]
    X_te0 = X_all[idx_te]

    if use_pca:
        n_comp = min(PCA_DIM, X_tr0.shape[0] - 1, X_tr0.shape[1])
        pca = PCA(n_components=n_comp, random_state=SEED)
        X_tr1 = pca.fit_transform(X_tr0)
        X_va1 = pca.transform(X_va0)
        X_te1 = pca.transform(X_te0)
    else:
        pca = None
        X_tr1, X_va1, X_te1 = X_tr0, X_va0, X_te0

    scaler = StandardScaler()
    X_tr2 = scaler.fit_transform(X_tr1).astype(np.float32)
    X_va2 = scaler.transform(X_va1).astype(np.float32)
    X_te2 = scaler.transform(X_te1).astype(np.float32)

    F = X_tr2.shape[1]
    X_full = np.zeros((N, F), dtype=np.float32)
    X_full[idx_tr] = X_tr2
    X_full[idx_va] = X_va2
    X_full[idx_te] = X_te2

    return X_full, X_tr2, X_va2, X_te2, pca

print("Raw feature dimension:", X_all.shape[1])
print("PCA feature dimension:", min(PCA_DIM, len(idx_tr)-1, X_all.shape[1]))


In [ ]:
# ======================
# FEATURE-BASED kNN GRAPH
# NO LABELS ARE USED HERE.
# ======================
def build_feature_knn_graph(X_full, k=K_GRAPH):
    nbrs = NearestNeighbors(
        n_neighbors=min(k + 1, len(X_full)),
        metric="cosine",
        n_jobs=-1
    )
    nbrs.fit(X_full)
    _, neigh = nbrs.kneighbors(X_full)

    rows, cols = [], []
    for i in range(len(X_full)):
        for j in neigh[i, 1:]:
            rows.append(i)
            cols.append(j)

    # Symmetric adjacency + self-loops
    rows2 = rows + cols + list(range(len(X_full)))
    cols2 = cols + rows + list(range(len(X_full)))
    data = np.ones(len(rows2), dtype=np.float32)

    A = coo_matrix(
        (data, (rows2, cols2)),
        shape=(len(X_full), len(X_full)),
        dtype=np.float32
    ).tocsr()

    A.data[:] = 1.0
    A_norm = gcn_filter(A)
    return A_norm

def graph_stats(A):
    A_csr = A.tocsr()
    edges_undirected = A_csr.nnz // 2
    avg_degree = A_csr.nnz / A_csr.shape[0]
    density = A_csr.nnz / (A_csr.shape[0] ** 2)
    return edges_undirected, avg_degree, density


In [ ]:
# ======================
# SIMPLE SOFTMAX HEAD
# Used for CNN and CNN + PCA ablations.
# ======================
def build_head(F):
    inp = Input(shape=(F,))
    x = layers.Dropout(0.30)(inp)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def evaluate_predictions(y_true, prob, idx_eval, name):
    pred = np.argmax(prob, axis=1)
    yt = y_true[idx_eval]
    yp = pred[idx_eval]

    acc = accuracy_score(yt, yp)
    macro_f1 = f1_score(yt, yp, average="macro")
    macro_p = precision_score(yt, yp, average="macro", zero_division=0)
    macro_r = recall_score(yt, yp, average="macro", zero_division=0)

    print(f"{name}: Accuracy={acc:.4f}, Macro-F1={macro_f1:.4f}")
    return {
        "Accuracy": acc,
        "Macro-F1": macro_f1,
        "Macro-Precision": macro_p,
        "Macro-Recall": macro_r
    }


In [ ]:
# ======================
# GCN MODEL
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,), name="X")
    A_in = Input(shape=(N,), sparse=True, name="A")

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model


In [ ]:
# ======================
# ABLATION 1: CNN ONLY
# ======================
# CNN-only baseline: the same CNN features are used without PCA or GCN.
X_raw, Xtr_raw, Xva_raw, Xte_raw, _ = make_features(use_pca=False)

head_cnn = build_head(Xtr_raw.shape[1])

head_cnn.fit(
    Xtr_raw, labels[idx_tr],
    validation_data=(Xva_raw, labels[idx_va]),
    epochs=EPOCHS_HEAD,
    batch_size=128,
    callbacks=[EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )],
    verbose=0
)

prob_va = head_cnn.predict(Xva_raw, verbose=0)
prob_te = head_cnn.predict(Xte_raw, verbose=0)

cnn_val_pred = np.argmax(prob_va, axis=1)
cnn_test_pred = np.argmax(prob_te, axis=1)

cnn_val_acc = accuracy_score(labels[idx_va], cnn_val_pred)
cnn_test_acc = accuracy_score(labels[idx_te], cnn_test_pred)

cnn_val_f1 = f1_score(labels[idx_va], cnn_val_pred, average="macro")
cnn_test_f1 = f1_score(labels[idx_te], cnn_test_pred, average="macro")

print(f"CNN validation accuracy: {cnn_val_acc:.4f}")
print(f"CNN test accuracy: {cnn_test_acc:.4f}")
print(f"CNN validation Macro-F1: {cnn_val_f1:.4f}")
print(f"CNN test Macro-F1: {cnn_test_f1:.4f}")


In [ ]:
# ======================
# HELPER: TRAIN A GCN ABLATION
# ======================
Y_cat = to_categorical(labels, num_classes).astype(np.float32)

def run_gcn_ablation(name, use_pca=True, weighted=True):
    X_full, Xtr, Xva, Xte, pca = make_features(use_pca=use_pca)

    # Graph is based only on features, never labels.
    A_norm = build_feature_knn_graph(X_full, k=K_GRAPH)

    model = build_gcn(Xtr.shape[1])

    if weighted:
        cw = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(num_classes),
            y=labels[idx_tr]
        )
        node_weights = np.zeros(N, dtype=np.float32)
        for c, w in enumerate(cw):
            node_weights[idx_tr[labels[idx_tr] == c]] = w
    else:
        node_weights = mask_tr.copy()

    val_weights = mask_va.copy()

    model.fit(
        [X_full, A_norm],
        Y_cat,
        sample_weight=node_weights,
        validation_data=([X_full, A_norm], Y_cat, val_weights),
        epochs=EPOCHS_GCN,
        batch_size=N,
        shuffle=False,
        callbacks=[EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True
        )],
        verbose=0
    )

    prob_all = model.predict([X_full, A_norm], batch_size=N, verbose=0)
    pred_all = np.argmax(prob_all, axis=1)

    val_acc = accuracy_score(labels[idx_va], pred_all[idx_va])
    test_acc = accuracy_score(labels[idx_te], pred_all[idx_te])

    val_f1 = f1_score(labels[idx_va], pred_all[idx_va], average="macro")
    test_f1 = f1_score(labels[idx_te], pred_all[idx_te], average="macro")

    report = classification_report(
        labels[idx_te],
        pred_all[idx_te],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    edges, avg_degree, density = graph_stats(A_norm)

    print(f"\n{name}")
    print(f"Feature dimension: {X_full.shape[1]}")
    print(f"Graph edges: {edges}")
    print(f"Average degree: {avg_degree:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Validation Macro-F1: {val_f1:.4f}")
    print(f"Test Macro-F1: {test_f1:.4f}")

    return {
        "Ablation": name,
        "Validation Accuracy": val_acc,
        "Test Accuracy": test_acc,
        "Validation Macro-F1": val_f1,
        "Test Macro-F1": test_f1,
        "Graph Edges": edges,
        "Average Degree": avg_degree,
        "Density": density,
        "Test Report": report
    }


In [ ]:
# ======================
# ABLATION 2: CNN + PCA
# ======================
X_pca, Xtr_pca, Xva_pca, Xte_pca, pca_model = make_features(use_pca=True)

head_pca = build_head(Xtr_pca.shape[1])

head_pca.fit(
    Xtr_pca, labels[idx_tr],
    validation_data=(Xva_pca, labels[idx_va]),
    epochs=EPOCHS_HEAD,
    batch_size=128,
    callbacks=[EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    )],
    verbose=0
)

pca_pred = np.argmax(head_pca.predict(Xte_pca, verbose=0), axis=1)

cnn_pca_test_acc = accuracy_score(labels[idx_te], pca_pred)
cnn_pca_test_f1 = f1_score(labels[idx_te], pca_pred, average="macro")

print(f"CNN + PCA test accuracy: {cnn_pca_test_acc:.4f}")
print(f"CNN + PCA test Macro-F1: {cnn_pca_test_f1:.4f}")


In [ ]:
# ======================
# ABLATION 3: GCN RAW
# ABLATION 4: GCN + PCA
# ABLATION 5: GCN + PCA WITHOUT CLASS WEIGHTING
# ======================
gcn_raw = run_gcn_ablation(
    "GCN Raw Features",
    use_pca=False,
    weighted=True
)

gcn_pca = run_gcn_ablation(
    "GCN + PCA",
    use_pca=True,
    weighted=True
)

gcn_pca_no_weight = run_gcn_ablation(
    "GCN + PCA without Class Weighting",
    use_pca=True,
    weighted=False
)


In [ ]:
# ======================
# FINAL ABLATION SUMMARY
# ======================
summary = pd.DataFrame([
    {
        "Ablation": "CNN",
        "Validation Accuracy": np.nan,
        "Test Accuracy": cnn_test_acc,
        "Validation Macro-F1": np.nan,
        "Test Macro-F1": cnn_test_f1
    },
    {
        "Ablation": "CNN + PCA",
        "Validation Accuracy": np.nan,
        "Test Accuracy": cnn_pca_test_acc,
        "Validation Macro-F1": np.nan,
        "Test Macro-F1": cnn_pca_test_f1
    },
    {k: v for k, v in gcn_raw.items() if k != "Test Report"},
    {k: v for k, v in gcn_pca.items() if k != "Test Report"},
    {k: v for k, v in gcn_pca_no_weight.items() if k != "Test Report"}
])

print(summary.round(4).to_string(index=False))


In [ ]:
# ======================
# PER-CLASS TEST PERFORMANCE
# ======================
# The run_gcn_ablation() function returns the classification report.
# Display the stored report for the proposed GCN + PCA configuration.

report_df = pd.DataFrame(gcn_pca["Test Report"]).T

print("\nGCN + PCA per-class test performance:")
print(
    report_df.loc[
        [c for c in class_names if c in report_df.index],
        ["precision", "recall", "f1-score", "support"]
    ].round(4).to_string()
)


## Interpretation

The corrected ablation should be interpreted only after execution. In particular, do **not** insert the previous 100% results into the manuscript. The key comparison is the held-out **test performance**:

1. **CNN vs. GCN** tests whether relational learning adds value beyond CNN features.
2. **GCN Raw vs. GCN + PCA** tests whether dimensionality reduction preserves discriminative information.
3. **GCN + PCA with vs. without class weighting** tests whether class weighting materially affects performance.
4. All GCN graphs are constructed from features rather than ground-truth labels, preventing the label leakage present in the previous domain-label graph implementation.

For the paper, report the actual values generated by this corrected notebook rather than estimated values.


In [ ]:
# ======================
# SAVE RESULTS
# ======================
summary.to_csv("hybrid_gcn_ablation_corrected_results.csv", index=False)
print("Saved: hybrid_gcn_ablation_corrected_results.csv")
